## Ch8-03 — Stale record detection

This notebook introduces stale record detection with `check_stale()`; after running it you can show how a stored content hash identifies records that need re-review when the model changes.


Evidence records are only as good as the model they reference. When a model changes, records hashed against the old source become stale — they cannot be trusted until re-reviewed. `check_stale()` in `evidence.py` detects this by comparing a ReviewRecord's `content_hash` against the current model source string. This notebook shows the full pattern: create a record, confirm it is current, change the model, confirm it becomes stale. See [Ch8-02 violation witness](02-violation-witness.ipynb) for the record structure.


In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    requirement def HeatingReq {
        subject heater : Heater;
        require constraint { heater.power >= 600.0 }
    }
    requirement heating : HeatingReq;
    part efficient : Heater;
    part weak : Heater { attribute :>> power = 400.0; }
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement {
        attribute resistance : Real default = 12.0;
    }
    part def PowerWire :> HeatingElement {
        attribute gauge : Real default = 14.0;
    }
    part def HeatingAssembly :> HeatingSystem {
        part coil : ResistanceCoil;
        part wire : PowerWire;
    }
    part heatingEvidence {
        assert satisfy heating by efficient;
        assert satisfy heating by weak;
    }
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    state Cycle {
        entry; then idle;
        state idle;
        state heating;
        state ready;
        state cancelled;
        transition first idle accept Start then heating;
        transition first heating accept Finish then ready;
        transition first heating accept Cancel then cancelled;
    }

    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")


In [ ]:
# A record with an empty identifier fails validation (not stale — invalid from the start).
from toaster.evidence import ReviewRecord, hash_content, validate_record

broken = ReviewRecord(
    identifier="",        # intentionally empty
    kind="asserted_solution",
    claim="Some claim",
    model_ref="ToasterDemo",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="Some criteria",
    rationale="Some rationale",
    counterevidence="Some counterevidence",
    record_kind="worked_example",
)
errors = validate_record(broken)
assert len(errors) > 0, "Expected validation errors for empty identifier"
print(f"Negative control ok: errors={errors}")


In [ ]:
from toaster.evidence import ReviewRecord, hash_content, check_stale

# Create a valid record hashed against the current model source
record = ReviewRecord(
    identifier="AS-C08-REV",
    kind="asserted_solution",
    claim="The nominal variant satisfies TimelyToast (cycleTime=120 ≤ 180).",
    model_ref="ToasterDemo::nominal",
    content_hash=hash_content(source),   # hash of current source
    scope="ToasterDemo",
    criteria="verify_satisfaction() returns holds=True for nominal",
    rationale=(
        "nominal.cycleTime=120 satisfies the constraint cycleTime ≤ 180. "
        "The attribute is set at the part definition level with no override."
    ),
    counterevidence=(
        "This uses a fixed cycleTime attribute. Real toasters vary with load. "
        "The claim is bounded to the model's defined operating conditions."
    ),
    residual_uncertainties="Thermal variability within a single cycle is not modelled.",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

# Confirm the record is current against the current source
assert not check_stale(record, source), "Record should be current"
print(f"Record is current: check_stale={check_stale(record, source)}")

# Simulate a model change: lower the requirement threshold
revised_source = source.replace(
    "toaster.cycleTime <= 180.0",
    "toaster.cycleTime <= 150.0",
)
revised_model = conn.load_from_content(revised_source, strict=False)
assert revised_model.ok, "Revised model should parse"

# Now check staleness against the revised source
stale = check_stale(record, revised_source)
assert stale, "Record should be stale after model change"
print(f"After constraint change: check_stale={stale}")
print("Record requires re-review: the stored hash no longer matches the current model.")
conn.close()


A ReviewRecord's `content_hash` bound to the original model source (A-F) is checked by `check_stale()` against the current model (O-S); changing the requirement threshold in the model source makes `check_stale()` return True, marking the record as stale (E).


Try the chapter exercise in `exercises/ch08/exercise.ipynb`: create a record for the HeatingReq satisfaction, change the minimum power threshold, and confirm that `check_stale()` fires.
